# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Lane 2 (Refresh / Content Opportunity Scoring) is a **scoring problem built on top of binary classification**. The end product is a ranked list — pages ordered by how urgently they deserve review — not a single label per page. But under the hood, the score is built from a classifier: predict the probability that a page is declining (or opportunity-worthy), then rank pages by that probability. This matches the starter pipeline exactly — `04_evaluate_and_export.py` combines a classifier's probability output with the baseline rule score into one `final_refresh_score`, then ranks.

It's not clustering (I'm not grouping pages into unlabeled archetypes) and it's not pure ranking-from-pairwise-comparisons (I'm not comparing pages head-to-head — I'm scoring each independently and sorting).


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Starter proxy target:** `is_declining_label = (trend_direction == "down")`.

This is a **defined-rule proxy**, not a future observed outcome — `trend_direction` is calculated from the same 90-day window I'd be using for features, comparing the last 30 days to the prior 30 days. It tells me "this page is currently trending down," not "this page will keep declining." I'm using it here in Week 2 because it's what the starter pipeline ships and it lets me frame the task concretely with real data today.

**Where I want this to go by the capstone:** a genuinely future-looking label — prior 90 days of features predicting decline (or recovery) over the *next* 30 days, built from the warehouse's daily fact table with a strict feature/target window split. That avoids the label secretly describing the same window the features come from.


In [1]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# same eligibility filter as the starter pipeline
eligible = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
eligible["is_declining_label"] = (eligible["trend_direction"] == "down").astype(int)

print("eligible pages:", len(eligible))
print(eligible["is_declining_label"].value_counts())
print(f"positive rate: {eligible['is_declining_label'].mean():.1%}")


eligible pages: 30000
is_declining_label
1    16262
0    13738
Name: count, dtype: int64
positive rate: 54.2%


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Precision@50** — of the top 50 pages my ranking puts at the top, how many are actually declining pages with real demand?

I'm choosing this over accuracy or plain AUC because the actual use case is capacity-limited: an editor reviews a fixed number of pages per sprint (I'm assuming ~50, matching what the starter pipeline reports), not the whole 30,000-page inventory. A model with great overall accuracy but a messy top-50 is useless here; a model with so-so overall accuracy but a clean top-50 is exactly what's needed. The starter pipeline already gives me a number to beat: baseline rule Precision@50 = 0.240, random forest Precision@50 = 0.740 (from `outputs/model_report.md`).


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [2]:
# unit of analysis: one row = one content page, described by its trailing
# 90-day search/engagement signals plus the proxy target column

unit_cols = [
    "content_id", "client_id",
    "impressions_90d", "sessions_90d", "clicks_90d",
    "avg_position", "ctr",
    "content_age_days", "days_since_last_update",
    "trend_direction", "trend_pct",
    "is_declining_label",
]

unit_of_analysis = eligible[unit_cols]
unit_of_analysis.head(10)


,content_id,client_id,impressions_90d,sessions_90d,clicks_90d,avg_position,ctr,content_age_days,days_since_last_update,trend_direction,trend_pct,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,17,29,10.6,0.76,187,20,down,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,15320,9,7,20.3,0.05,445,25,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,11,36.5,0.09,141,20,down,-60.9,1
3,content_331d6c4de07b,client_19581e27de,11751,78,58,6.2,0.49,463,22,stable,-13.8,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,145,24,44.0,0.13,263,14,down,-34.7,1
5,content_d4084a4bc775,client_f369cb89fc,3970,5,1,8.5,0.03,147,20,down,-38.9,1
6,content_9a34b442b552,client_8722616204,20,1,0,7.0,0.00,90,20,down,-92.3,1
7,content_a63219c6e95a,client_19581e27de,1724,28,1,21.2,0.06,445,22,stable,0.6,0
8,content_5e6c160719bc,client_6208ef0f77,32574,68,29,46.0,0.09,90,20,down,-58.8,1
9,content_c27558df2b0c,client_19581e27de,1240,3,2,4.9,0.16,257,104,down,-29.2,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

The baseline rule (`stale_visible_page`, `declining_with_demand`, etc.) each check one or two conditions in isolation — e.g. "trend is down AND impressions >= 100." But whether a page is actually worth a reviewer's time depends on many signals interacting at once: position, CTR relative to that position, content age, freshness, word count, and how those combine differs across content types and clients. A hand-written if/else can't easily capture something like "pages ranking 8-15 with below-tier CTR and moderate freshness decay tend to be worth it, but only when word count is already adequate" — that's a multi-way interaction, and there could be dozens of them.

The starter pipeline's own numbers make this concrete: the fixed baseline rule gets Precision@50 = 0.240 (about 12 of the top 50 picks are right), while a random forest trained on the same signals gets Precision@50 = 0.740 (about 37 of 50 right) — roughly 3x better, on the same data, using signals the rule already had access to. That gap is the model finding interactions the rule's simple thresholds miss.


In [3]:
# Numbers already established in w01 and reused here as the evidence base:
baseline_precision_at_50 = 0.240
random_forest_precision_at_50 = 0.740

lift = random_forest_precision_at_50 / baseline_precision_at_50
print(f"baseline rule Precision@50: {baseline_precision_at_50}")
print(f"random forest Precision@50: {random_forest_precision_at_50}")
print(f"lift: {lift:.2f}x")


baseline rule Precision@50: 0.24
random forest Precision@50: 0.74
lift: 3.08x


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.